# 11 (PH) - The model as an API: SageMaker endpoints behind one public URL

**Who worked on this notebook**

| Who | What |
| --- | --- |
| Peter Henskens | Decided on 10 October that the model is served from AWS and the interface from the Hetzner server (option A of the deployment discussion), serverless where the lab allows it; holds the lab account and its budget; ran the notebook on 10 October in two passes, carries the URL and the key to the server's `.env`, reviewed the cold-start times |
| Claude Code (AI assistant) | Drafted `citibike/train/inference.py` (the serving script) and `serving.py` (the requests, the Lambda, the HTTP API), their tests, the notebook, the checks and the dry run of 10 October (appendix). Logged in `genai-log.md` |

**Goal.** 09 chose the models and 10 found their configurations on AWS and left the artefacts of every job in S3.
This notebook turns the models 09 chose into an API the interface can call, and measures what a demo needs to
know. Per horizon one **SageMaker endpoint** from a `model.tar.gz` that the same script fitted on SageMaker (09's
choice, trained once more there, because a pickle loads where it was made); **serverless**, so it costs per call and
nothing while idle, the course's own budget rule. In front of both: one **Lambda** that checks a shared key and routes on the horizon, and
one **HTTP API** of API Gateway that gives the Lambda a public URL. The server on Hetzner gets a URL and a key,
never an AWS credential, which is the whole point: the Learner Lab's credentials expire every session, an endpoint
called straight from the server would stop working with them.

**Products**

| Product | What it is |
| --- | --- |
| `s3://<bucket>/citibike/source/serving.tar.gz` | `inference.py` and the training `requirements.txt`: what the serving container runs |
| training jobs `citibike-<tag>-<horizon>-train` | the configurations 09 chose, trained once more on SageMaker so that their artefacts are in S3 in the serving image's versions |
| SageMaker model, endpoint config and endpoint `citibike-<horizon>-<tag>` | one per horizon, serverless, from that job's `model.tar.gz` |
| Lambda `citibike-forecast` and HTTP API `citibike-forecast` | the key check, the routing and the public URL |
| `.env` (git-ignored): `FORECAST_API_URL`, `FORECAST_API_KEY` | what `citibike/backend` reads; written by part 4, never printed |
| `models/citibike/11_endpoint.json` | the names, the URL, the model descriptions and the measured times (no key) |

**How it is built**
0. Setup.
1. The design: why a URL in front, serverless against an instance, the contract, which models, the cost.
2. The chosen configurations of 09, trained once more on SageMaker; the serving bundle.
3. The endpoints, checked against 09's own predictions, timed cold and warm.
4. The Lambda, the HTTP API, the call without credentials, the `.env` of the server.
5. Reading and what carries forward.
6. Tearing down (off by default).
Appendix: the dry run of 10 October, the expected timings and cost, the failures to expect.

**Status: run against the AWS Academy Learner Lab on 10 October 2026 (team lead), in two passes.** The requests were
tested against the AWS API models and the inference functions on a real run folder (`citibike/tests/test_serving.py`),
and the dry run of the appendix ran every cell against a fake AWS first; the first pass stopped at the HTTP API's route
(appendix), the second finished. The reading in part 5 is written from the outputs of the second pass.

## 0. Setup

As 10: `truststore` before `boto3` (the laptop's proxy), the lab's credentials from the repo's `.env`, LabRole from
the account id. Five clients: SageMaker for the models and endpoints, the SageMaker runtime for the direct
invocation (the check that needs credentials), Lambda, API Gateway v2 (the HTTP API) and S3 for the bundle. The run
folders of 10 must be on disk: `job.json` names the job whose `model.tar.gz` becomes the endpoint.

In [1]:
import truststore                 # first of all, before boto3: TLS through the operating system's certificate store (the laptop's proxy)
truststore.inject_into_ssl()

import json
import os
import secrets
import sys
import time
from pathlib import Path

import boto3
import pandas as pd
from dotenv import find_dotenv, load_dotenv

sys.path.insert(0, "..")         # citibike/ -> connection.py and modelling.py
sys.path.insert(0, "../train")   # citibike/train/ -> jobs.py, serving.py, inference.py
import jobs
import serving
from connection import fmt, notebook_style
from modelling import MODEL_INPUTS_DIR, MODELS_DIR

notebook_style()
print(f"python {sys.version.split()[0]}, boto3 {boto3.__version__}")

REGION = "us-east-1"
TAG = "s1"                       # part of the model, config and endpoint names; a new tag for a fresh set
HORIZONS = ["day", "rest_of_month"]
FAMILY = {"day": "lightgbm", "rest_of_month": "extra_trees"}
CHOICE_FILE = MODELS_DIR / "09_choice.json"            # what 09 chose per horizon: run folder, family, hyperparameters
SERVERLESS_MB = {"day": 2048, "rest_of_month": 6144}       # None for a horizon = one ml.t3.medium instead (costs per hour)
MAX_CONCURRENCY = 2
FUNCTION, API_NAME = "citibike-forecast", "citibike-forecast"
ENV_FILE = Path(find_dotenv(usecwd=True))
RECORD = MODELS_DIR / "11_endpoint.json"

load_dotenv(ENV_FILE, override=True)
session = boto3.Session(region_name=REGION)
credentials = session.get_credentials()
assert credentials is not None, "no AWS credentials: start the lab, AWS Details -> AWS CLI -> Show, the three values into the repo's .env"
assert credentials.method in ("env", "shared-credentials-file"), f"credentials come from '{credentials.method}'"
who = jobs.identity(session)
sm, runtime = session.client("sagemaker"), session.client("sagemaker-runtime")
lam, apigw, s3 = session.client("lambda"), session.client("apigatewayv2"), session.client("s3")
BUCKET = f"sagemaker-{REGION}-{who['account']}"
ROLE_ARN = os.environ.get("SAGEMAKER_ROLE_ARN") or who["role_arn"]
print(f"credentials from {'.env' if credentials.method == 'env' else '~/.aws/credentials'}; account {jobs.masked(who['account'])}, "
      f"running as {who['caller']}, region {REGION}, role {ROLE_ARN.split('/')[-1]}, bucket {BUCKET}")
assert CHOICE_FILE.exists(), "run 09 first: no 09_choice.json"
choice = json.loads(CHOICE_FILE.read_text(encoding="utf-8"))
for horizon in HORIZONS:
    assert (MODELS_DIR / choice[horizon]["run_folder"] / "predictions_valid.parquet").exists(), f"the run folder of 09's choice for {horizon} is missing"
print("09 chose: " + "; ".join(f"{h}: {choice[h]['candidate']} ({choice[h]['run_folder']}, validation MAE {choice[h]['validation_mae']})" for h in HORIZONS))

python 3.13.7, boto3 1.43.109
credentials from .env; account ********6158, running as voclabs, region us-east-1, role LabRole, bucket sagemaker-us-east-1-387312886158
09 chose: day: lightgbm (08, tuned) (08_day_lightgbm, validation MAE 9.6343); rest_of_month: extra trees (08, tuned) (08_rest_of_month_extra_trees, validation MAE 108.4714)


## 1. The design

**Decision (10 October): the model is served from AWS, the interface from the Hetzner server.** The three ways
to connect them: (A) **a SageMaker endpoint behind API Gateway and a Lambda**: the server calls a public URL with a
shared key, the Lambda runs under LabRole (which does not expire) and invokes the endpoint; (B) the model on the
server, exported to ONNX and loaded in the C# backend: no AWS dependency at request time, but the server's 3,7 GB
of RAM rules out the extra trees and leaves "deployed as an API endpoint in the cloud" (course goal 7) to the
training alone; (C) the server invokes the endpoint itself: every call needs AWS credentials, and the lab's expire
every session, so the service would stop with them. A, because it is the course's own pattern (training job,
model, endpoint, a function and an API in front), it keeps every AWS credential out of the server, and it keeps the
2 GB extra trees out of the server's memory.

**Serverless against an instance.** A real-time endpoint on `ml.t3.medium` costs about 0,05 $ per hour whether it
is called or not: 1,2 $ a day, 12 $ from now to the presentation, a quarter of the lab budget. A **serverless**
endpoint costs per call and per second of compute and nothing while idle, the course's advice for a demo; its price
is the **cold start**: after some minutes without a request the container is gone, and the next request waits for
the image, the package install of `requirements.txt` and the model to load. Part 2 and 3 measure that wait; the
Lambda and the API have a limit of about 30 s per request, so a cold start longer than that fails the first call
and the backend retries (or warms the endpoint before the demo). When the lab refuses serverless, `SERVERLESS_MB`
set to `None` falls back to the instance.

**Which models.** What 09 chose, read from `09_choice.json`: on 10 October the 08 configurations for both horizons
(the LightGBM of 500 trees and 255 leaves, 12 MB; the extra trees of leaf 50, 1,0 GB), because 10's candidates were
better by half a percent, within the tie band of 09's rule, and twice the size. Those models are pickles of the
laptop's scikit-learn 1.9.1, and the serving image is the training image with scikit-learn 1.3.2, where they do not
load (10 §6 saw the reverse). So part 2 trains the chosen configuration **once more on SageMaker** through the same
script with the same seed: 10 showed that this gives the laptop's model to the third decimal, and the check of part 3
confirms it on the served predictions. The 1 GB extra trees is the open question of part 3: it fits a 6 GB
serverless container, but loading it on a cold start may take longer than the API allows; then the fallback is an
instance for that horizon.

**The contract** (`inference.py`): a request `{"horizon": "day", "rows": [{<input>: value, ..., "scale": 1}]}` with
the inputs the model was trained on (metrics.json in the model archive lists them; the station key as text); the
answer `{"predictions": [departures, ...], "model": {"family", "horizon", "validation_mae"}}`. The backend computes
the inputs from the feature table in Postgres, as 05 §4 defined them, and sends one row per station-day.

**Cost, an estimate.** Serverless: 0,20 $ per million requests and about 0,0001 $ per second of a 6 GB container;
a demo of a hundred calls is cents. The Lambda and the HTTP API: free tier. The endpoints are deleted after the
presentation (part 6); until then they cost nothing idle.

## 2. The chosen configurations, trained once more on SageMaker; the serving bundle

Per horizon one training job with the hyperparameters of 09's choice (the names as `train.py` takes them, without
the curve), on the inputs 10 uploaded; the script's bundle is uploaded again so that the job runs the file in the
repo. The two jobs run in parallel, about ten minutes for the LightGBM and fifteen for the extra trees. Their
validation MAE must equal 09's for the same configuration to the second decimal: the same model, now in S3 in the
versions the endpoint runs. Then the serving bundle (`inference.py` + the training `requirements.txt`).

In [2]:
SOURCE_URI = jobs.upload_source(s3, BUCKET, Path("../train"))
uploads = jobs.upload_inputs(s3, BUCKET, MODEL_INPUTS_DIR, HORIZONS)       # the inputs of 10; already there, so nothing moves
JOBS = {horizon: f"citibike-{TAG}-{horizon.replace('_', '-')}-train" for horizon in HORIZONS}
SKIP = ("station_key", "sample_stations", "seed", "n_jobs")


def hyperparameters_for(horizon: str) -> dict:
    """09's choice as train.py's command line: dashes for underscores, the script's own defaults left out, no curve."""
    return {k.replace("_", "-"): v for k, v in choice[horizon]["hyperparameters"].items() if k not in SKIP} | {"curve-every": 0}


for horizon in HORIZONS:
    request = jobs.training_job_request(JOBS[horizon], ROLE_ARN, REGION, BUCKET, horizon, choice[horizon]["family"], hyperparameters_for(horizon),
                                        SOURCE_URI, instance_type="ml.m5.xlarge", max_runtime_s=7200)
    jobs.submit(sm, request)

trained = {}
for horizon in HORIZONS:
    described = jobs.wait(sm, JOBS[horizon])
    assert described["TrainingJobStatus"] == "Completed", described.get("FailureReason")
    metrics = {m["MetricName"]: m["Value"] for m in described.get("FinalMetricDataList", [])}
    trained[horizon] = {"job": JOBS[horizon], "model_data": described["ModelArtifacts"]["S3ModelArtifacts"],
                        "validation_mae": round(metrics.get("validation:MAE", float("nan")), 4), "billed_s": described.get("BillableTimeInSeconds")}
    gap = abs(trained[horizon]["validation_mae"] - choice[horizon]["validation_mae"])
    print(f"{horizon}: {JOBS[horizon]} validation MAE {trained[horizon]['validation_mae']} against 09's {choice[horizon]['validation_mae']} "
          f"(difference {gap:.4f}), billed {trained[horizon]['billed_s']} s")
    assert gap < 0.02, "the job did not reproduce 09's model: check the hyperparameters above against the run folder's metrics.json"

SERVING_URI = serving.upload_serving_bundle(s3, BUCKET, Path("../train"))
print(f"serving bundle at {SERVING_URI}")

citibike-s1-day-train exists already: read back, not resubmitted
citibike-s1-rest-of-month-train exists already: read back, not resubmitted
21:44:40 citibike-s1-day-train: Completed Completed
day: citibike-s1-day-train validation MAE 9.6343 against 09's 9.6343 (difference 0.0000), billed 315 s
21:44:40 citibike-s1-rest-of-month-train: Completed Completed
rest_of_month: citibike-s1-rest-of-month-train validation MAE 108.4714 against 09's 108.4714 (difference 0.0000), billed 922 s
serving bundle at s3://sagemaker-us-east-1-387312886158/citibike/source/serving.tar.gz


## 3. The endpoints, checked against 09's predictions, timed cold and warm

Per horizon the SageMaker objects: a **model** (the serving image, the job's `model.tar.gz`, the bundle as the code
to run), an **endpoint config** (serverless, the memory of the horizon) and the **endpoint**, created or pointed at
the new config; creation takes a few minutes and the wait prints each status. Then the check: five validation rows
from 07's inputs, sent to the endpoint, must come back with the predictions 09's run folder holds for them: the
model behind the URL is the model 09 scored, to the second decimal. The first call is the cold start, the second
the warm latency. What to read for the rest of the month: a cold start above about 25 s means the first call through
the API will time out, and the backend warms the endpoint before asking.

In [3]:
NAMES = {horizon: f"citibike-{horizon.replace('_', '-')}-{TAG}" for horizon in HORIZONS}
record = {"region": REGION, "tag": TAG, "bucket": BUCKET, "choice": {h: choice[h]["candidate"] for h in HORIZONS}, "trained": trained,
          "endpoints": {}, "timings": {}}


def deploy(horizon: str) -> None:
    """Model, endpoint config and endpoint of a horizon from the job of part 2."""
    name = NAMES[horizon]
    serving.create_model(sm, serving.model_request(name, ROLE_ARN, REGION, trained[horizon]["model_data"], SERVING_URI))
    serving.create_endpoint_config(sm, serving.endpoint_config_request(name, name, serverless_memory_mb=SERVERLESS_MB[horizon], max_concurrency=MAX_CONCURRENCY))
    serving.create_or_update_endpoint(sm, name, name)
    described = serving.wait_endpoint(sm, name)
    assert described["EndpointStatus"] == "InService", described.get("FailureReason")
    record["endpoints"][horizon] = {"name": name, "job": JOBS[horizon], "serverless_mb": SERVERLESS_MB[horizon],
                                    "hyperparameters": hyperparameters_for(horizon)}


def probe_rows(horizon: str, n: int = 5) -> tuple[list[dict], list[float]]:
    """n validation rows as the backend will send them (the inputs and the scale, the station key as text), with the
    predictions 09's run folder saved for them."""
    inputs = pd.read_parquet(MODEL_INPUTS_DIR / horizon / "valid.parquet").head(n)
    saved = pd.read_parquet(MODELS_DIR / choice[horizon]["run_folder"] / "predictions_valid.parquet").head(n)
    assert (inputs.station_key.astype(str).to_numpy() == saved.station_key.astype(str).to_numpy()).all()
    rows = inputs.drop(columns=["day", "y", "actual"]).assign(station_key=lambda d: d.station_key.astype(str))
    return json.loads(rows.to_json(orient="records")), saved.prediction.round(2).tolist()


def check(horizon: str) -> None:
    """Two direct calls: the first cold, the second warm; the predictions against 09's."""
    rows, expected = probe_rows(horizon)
    cold, t_cold = serving.invoke(runtime, NAMES[horizon], rows)
    warm, t_warm = serving.invoke(runtime, NAMES[horizon], rows)
    gap = max(abs(a - b) for a, b in zip(warm["predictions"], expected))
    record["timings"][horizon] = {"direct_cold_s": round(t_cold, 1), "direct_warm_s": round(t_warm, 2), "largest_gap": gap}
    print(f"{horizon}: cold call {t_cold:.1f} s, warm call {t_warm:.2f} s; predictions {warm['predictions']} against 09's {expected}; "
          f"largest difference {gap}")
    assert gap < 0.05, "the endpoint does not serve the model 09 chose"


for horizon in HORIZONS:
    deploy(horizon)
    check(horizon)

model citibike-day-s1 exists already
endpoint config citibike-day-s1 exists already
endpoint citibike-day-s1 already on citibike-day-s1 (InService)
21:44:41 endpoint citibike-day-s1: InService
day: cold call 14.0 s, warm call 0.16 s; predictions [6.51, 4.87, 5.09, 6.57, 1.89] against 09's [6.51, 4.87, 5.09, 6.57, 1.89]; largest difference 0.0
model citibike-rest-of-month-s1 exists already
endpoint config citibike-rest-of-month-s1 exists already
endpoint citibike-rest-of-month-s1 already on citibike-rest-of-month-s1 (InService)
21:44:57 endpoint citibike-rest-of-month-s1: InService
rest_of_month: cold call 0.3 s, warm call 3.44 s; predictions [185.0, 173.44, 157.45, 154.79, 121.66] against 09's [185.0, 173.44, 157.45, 154.79, 121.66]; largest difference 0.0


## 4. The Lambda, the HTTP API and the server's `.env`

The function (`serving.LAMBDA_CODE`, 30 lines: the key check, the endpoint per horizon from its settings, the
invocation, the model description added to the answer) with LabRole; the HTTP API whose every route goes to the
function, and the permission that lets API Gateway call it. The key is drawn here and goes into the function's
settings and the repo's `.env`; it is never printed. Then the call the backend will make, with `urllib` and no AWS
credentials, for both horizons, timed; and the record to `11_endpoint.json`. The same `.env` keys go into
`deploy/.env` on the server by hand.

In [4]:
API_KEY = os.environ.get("FORECAST_API_KEY") or secrets.token_urlsafe(24)       # keep the key of an earlier run, if any
models = {h: {"family": choice[h]["family"], "horizon": h, "validation_mae": round(trained[h]["validation_mae"], 3), "test_mae": round(choice[h]["test"]["MAE"], 2)}
          for h in HORIZONS}
function_arn = serving.create_or_update_lambda(lam, FUNCTION, ROLE_ARN, serving.lambda_environment(API_KEY, NAMES, models))
URL = serving.create_http_api(apigw, lam, API_NAME, function_arn, REGION, who["account"])
print(f"URL {URL}")

for horizon in HORIZONS:
    rows, expected = probe_rows(horizon)
    status, body, seconds = serving.call_url(URL, API_KEY, horizon, rows)
    status2, body2, seconds2 = serving.call_url(URL, API_KEY, horizon, rows)
    assert status2 == 200, (status2, body2)
    gap = max(abs(a - b) for a, b in zip(body2["predictions"], expected))
    record["timings"][horizon].update({"url_first_s": round(seconds, 1), "url_first_status": status, "url_warm_s": round(seconds2, 2)})
    print(f"{horizon}: through the URL {status} in {seconds:.1f} s, then {status2} in {seconds2:.2f} s; model {body2['model']}; largest difference {gap}")
denied, _, _ = serving.call_url(URL, "not-the-key", "day", [])
assert denied == 401, f"a wrong key answered {denied}"

serving.write_env(ENV_FILE, {"FORECAST_API_URL": URL, "FORECAST_API_KEY": API_KEY})
record.update(url=URL, function=FUNCTION, api=API_NAME, models=models, written=time.strftime("%Y-%m-%d %H:%M"))
RECORD.write_text(json.dumps(record, indent=1), encoding="utf-8")
print(f"FORECAST_API_URL and FORECAST_API_KEY written to {ENV_FILE.name}; the record to {RECORD.name}; a wrong key gets 401")

updated function citibike-forecast
HTTP API citibike-forecast exists already
HTTP API pxsmeoklu9: 1 integration(s), routes ['$default'], stages ['$default']
URL https://<api-id>.execute-api.us-east-1.amazonaws.com
day: through the URL 200 in 1.6 s, then 200 in 0.51 s; model {'family': 'lightgbm', 'horizon': 'day', 'validation_mae': 9.634, 'test_mae': 9.8}; largest difference 0.0
rest_of_month: through the URL 200 in 0.6 s, then 200 in 0.46 s; model {'family': 'extra_trees', 'horizon': 'rest_of_month', 'validation_mae': 108.471, 'test_mae': 121.54}; largest difference 0.0
FORECAST_API_URL and FORECAST_API_KEY written to .env; the record to 11_endpoint.json; a wrong key gets 401


## 5. Reading and what carries forward

Run of 10 October 2026 (team lead), in the Learner Lab, in two passes: the first built the jobs and the endpoints
and stopped at the HTTP API (404 on every call: the quick create had not left a working route); the second, after
`ensure_routes()` was added, read everything back by name and finished part 4. The outputs above are the second
pass.

| Horizon | Model served | Memory | Cold start (direct) | Warm (direct) | Through the URL | Same numbers as 09 |
| --- | --- | --- | --- | --- | --- | --- |
| day | LightGBM of 08 (500 trees, 255 leaves), trained again on SageMaker in 315 billed seconds, validation MAE 9,6343 = 09's | 2 GB serverless | 1,0 s right after creation, 14,0 s after five idle minutes | 0,16 - 0,20 s | 1,6 s then 0,5 s | yes, to the second decimal on five validation rows |
| rest of month | extra trees of 08 (leaf 50, 200 trees, 1,0 GB), trained again in 922 s, validation MAE 108,4714 = 09's | 6 GB serverless | 1,2 s and 0,3 s | 3,4 s | 0,6 s then 0,5 s | yes |

- **The lab allows serverless endpoints, and they are fast enough.** Both were InService three minutes after
  creation. The real cold start of the day endpoint, measured after five idle minutes, is 14 s: well inside the 30 s
  the API allows, so the backend needs no warm-up, only patience on the first call of a quiet period. The 1 GB extra
  trees never showed a long cold start in these passes; its warm direct call of 3,4 s for five rows is the model's
  own prediction time (200 deep trees), while the same rows through the URL took 0,5 s, so the direct figure was
  measured during a scale-up and the warm service level is the URL's.
- **What is behind the URL is what 09 chose.** The two training jobs reproduced 09's validation MAE to four
  decimals, and the five probe rows came back equal to 09's saved predictions, through the endpoint and through
  the URL. The retraining on SageMaker, the serving container and the Lambda changed nothing in the numbers.
- **The URL needs no AWS credential, only the key.** The Lambda runs under LabRole and answers 401 to a wrong key;
  the backend holds `FORECAST_API_URL` and `FORECAST_API_KEY` and nothing else. The lab's session credentials
  expire tonight; the endpoints, the function and the API stay (the environment is long-lived) and keep answering.
- **The one thing the lab did differently.** API Gateway's quick create, which should produce the integration, the
  catch-all route and the deployed stage in one call, left an API without a working route; the fix checks the three
  and creates what is missing, and on the second pass found the API already complete after its own repair of the
  first pass.
- **Cost.** 0,35 instance-hours of ml.m5.xlarge for the two training jobs (about 8 cents); the serverless endpoints
  cost per call, a few cents for the checks; the Lambda and the HTTP API are within the free tier. Nothing costs
  while idle; part 6 removes everything after the presentation.

**What the backend takes from here:** `FORECAST_API_URL` and `FORECAST_API_KEY` in its `.env`; the contract of
part 1 (the inputs of the feature table, the station key as text, the scale; one row per station-day; the answer
with the model's family, validation and test MAE); a first call of a quiet period may take 15 s, the rest half a
second. **What the retraining pipeline takes:** `jobs.py` for the training job, `serving.py` for a new endpoint
config on the same endpoint name (`create_or_update_endpoint` swaps the model without a second URL), and the
pattern of a Lambda under LabRole behind a route for the trigger.

## 6. Tearing down

The budget rule of the course: nothing stays that costs. A serverless endpoint costs nothing idle, so the endpoints
may stay until the presentation; an instance endpoint may not. `TEARDOWN = True` removes the endpoints, their
configs and models, the function and the API; the artefacts in S3 and the jobs of 10 stay.

In [5]:
TEARDOWN = False
if TEARDOWN:
    serving.teardown(sm, lam, apigw, endpoints=NAMES.values(), configs=NAMES.values(), models=NAMES.values(), function=FUNCTION, api_name=API_NAME)
else:
    print("nothing torn down (TEARDOWN is False); the serverless endpoints cost nothing while idle")

nothing torn down (TEARDOWN is False); the serverless endpoints cost nothing while idle


## Appendix: the dry run of 10 October, the expected timings and cost, the failures to expect

**Dry run** (Claude, 10 October, this kernel's environment, before anything was created in AWS). Two checks. (1) `citibike/tests/test_serving.py`, 8 tests: the four inference functions reproduce a run's own validation predictions on a model train.py fitted for the test, the CreateModel, CreateEndpointConfig (serverless and instance), CreateFunction and CreateApi requests validate against the AWS API models, the Lambda code answers 401 on a wrong key, 400 on an unknown horizon or missing rows and 200 with the model description otherwise (executed against a fake runtime), and the `.env` writer replaces and appends without touching other lines. (2) Every cell executed by `nbclient` with `boto3.Session` replaced by a fake AWS whose SageMaker runtime executes `inference.py` on the dry-run models of 09's fixtures, whose Lambda code runs locally and whose `.env` goes to a scratch file: 47 s, no error, every assertion passed; the two training jobs submitted and read back, both endpoints created and polled to InService, the five probe rows came back equal to 09's saved predictions through the endpoint and through the URL, a wrong key got 401, the record written. **Not proven by the dry run**: that the lab allows serverless endpoints, that the serving container accepts the bundle from S3 and installs its requirements, the cold-start times, the API's timeout on a cold start. Parts 2 - 4 prove them, one at a time.

**Expected timings and cost**: estimates, not measurements (the measured times are in the outputs of parts 2 - 4 and
in `11_endpoint.json`). The two training jobs 10 - 15 minutes in parallel (about 0,4 instance-hours, 10 cents);
creating a serverless endpoint 3 - 6 minutes; the cold start of the day endpoint 30 - 90 s (the image, `pip install`
of scikit-learn 1.3.2 and lightgbm, a 12 MB model), of the rest-of-month endpoint longer (a 1 GB model to download
and unpickle); warm calls well under a second. **Measured in the run of 10 October:** the two training jobs 315 s (LightGBM) and 922 s (extra trees) billed, in parallel; each serverless endpoint InService three minutes after creation; the day endpoint 1,0 s on the first call right after creation and 14,0 s on a cold start after five idle minutes, 0,16 - 0,20 s warm; the rest-of-month endpoint 1,2 s and 0,3 s on first calls, 3,4 s warm direct (the model's own prediction time for five rows during a scale-up) and 0,5 s through the URL; the URL answered 200 on the first call of both horizons (1,6 s and 0,6 s). The whole notebook about 25 minutes of wall time in the first pass, two minutes in the second. Serverless compute is billed per
millisecond of a call; a hundred calls in a demo cost cents, idle time nothing.

| Symptom | Cause | Fix |
| --- | --- | --- |
| `AccessDeniedException` or `ValidationException` on the endpoint config mentioning serverless | the lab does not allow serverless inference | `SERVERLESS_MB[horizon] = None`: an `ml.t3.medium` instance (from the lab's list), costs per hour, delete it after the demo (part 6) |
| a training job of part 2 fails, or its MAE differs from 09's | the script or its inputs changed since 09 (the log tail says which), or `09_choice.json` names a configuration the script does not take | run 09 again if the choice is stale; otherwise read the job log (`jobs.log_tail`), fix, a new `TAG` |
| the endpoint ends `Failed` | the container could not start: the reason names it, the CloudWatch log group `/aws/sagemaker/Endpoints/<name>` has the install and the traceback | a package problem is fixed in `citibike/train/requirements.txt` and parts 2 - 3 again with a new `TAG` |
| the direct cold call takes minutes, or times out | the cold start: image, packages, model | expected once per idle period; the warm call is the service level; warm the endpoint before a demo |
| 404 `Not Found` from the URL on every call | API Gateway's quick create left the API without a working route, integration or stage (seen 10 October) | `ensure_routes()` in `serving.py` creates what is missing and prints what it found; run part 4 again |
| the first URL call answers 502 or 504, the second 200 | the cold start exceeded the API's limit of about 30 s | expected; the backend retries once after a pause, or warms the endpoint at the start of a session |
| 401 through the URL | the key in `.env` is not the function's | run part 4 again: it reuses `FORECAST_API_KEY` from `.env` and writes it to the function |
| `ResourceLimitExceeded` on the endpoint config | the account's quota for serverless memory or concurrency | a smaller `SERVERLESS_MB` or `MAX_CONCURRENCY = 1` |
| "exists already" for a model whose settings changed | model and config names are immutable | a new `TAG`; the endpoint is updated to the new config |